In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -U chronos-forecasting -q

In [ ]:
import numpy as np
import torch
from chronos import Chronos2Pipeline

SAVE_DIR = "/content/drive/MyDrive/diss-datasets"
device = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', device)

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
)
print('Chronos-2 loaded')

In [ ]:
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient.npy', allow_pickle=True).item()
X_tr = data['train']['X']
print(X_tr.shape, X_tr.dtype)

w = X_tr[0].astype(np.float32).copy()
assert w.shape == (18, 2560), w.shape
print(w.shape, w.min(), w.max())

In [ ]:
def extract_raw(window):
    context = torch.tensor(window[None, :, :], dtype=torch.float32)
    print('  passed to model:', context.shape)
    embeddings, _ = pipeline.embed(context)
    emb = embeddings[0]
    print('  returned:', emb.shape)
    return emb.detach().cpu().numpy()

In [ ]:
rng = np.random.default_rng(0)

w1 = w.copy()
w2 = w.copy()
w2[0] = rng.normal(0.0, 1.0, size=w.shape[1]).astype(np.float32)

print('input delta ch0 :', np.abs(w1[0] - w2[0]).max())
print('input delta rest:', np.abs(w1[1:] - w2[1:]).max())

In [ ]:
print('run 1')
E1 = extract_raw(w1)
print('run 2')
E2 = extract_raw(w2)
print('run 3')
E3 = extract_raw(w1)

In [ ]:
determinism = np.abs(E1 - E3).max()
ch0_delta   = np.abs(E1[0]  - E2[0]).max()
rest_delta  = np.abs(E1[1:] - E2[1:]).max()

print(f'determinism floor : {determinism:.3e}')
print(f'channel 0 delta   : {ch0_delta:.3e}')
print(f'channels 1-17     : {rest_delta:.3e}')
print()

if ch0_delta <= determinism:
    print('BROKEN: model is not reading the input')
elif rest_delta <= determinism:
    print('GROUP ATTENTION OFF: channels processed independently')
else:
    print(f'GROUP ATTENTION ON: {rest_delta / max(determinism, 1e-12):.1f}x above noise floor')

In [ ]:
import json, datetime

result = {
    'timestamp': datetime.datetime.now().isoformat(),
    'determinism_floor': float(determinism),
    'channel_0_delta': float(ch0_delta),
    'channels_1_17_delta': float(rest_delta),
    'embedding_shape': list(E1.shape),
}

with open(f'{SAVE_DIR}/chronos2_channel_leakage_test.json', 'w') as f:
    json.dump(result, f, indent=2)

print(json.dumps(result, indent=2))

In [ ]:
w3 = w.copy()
w3[9] = rng.normal(0.0, 1.0, size=w.shape[1]).astype(np.float32)
E4 = extract_raw(w3)

other_delta = np.abs(np.delete(E1, 9, axis=0) - np.delete(E4, 9, axis=0)).max()
print(f'channel 9 corrupted, effect on others: {other_delta:.3e}')

In [ ]:
result['channel_9_delta'] = float(other_delta)

with open(f'{SAVE_DIR}/chronos2_channel_leakage_test.json', 'w') as f:
    json.dump(result, f, indent=2)